<a href="https://colab.research.google.com/github/awinarko-hue/Project_hue/blob/main/Legal/Ekstrak/Exp01_R2_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exp01_R2_versi_2 - Konstruksi Knowledge Graph Struktur Norma & Audit Akurasi Rujukan (RQ1)

**Pertanyaan penelitian (RQ1):** apakah graf pengetahuan regulasi telekomunikasi Indonesia yang merepresentasikan hierarki, rujukan silang antarpasal yang tervalidasi, dan rujukan antardokumen dapat dibangun secara otomatis, **akurat**, dan **dapat direproduksi**?

**Alur notebook**
1. Tahap 1 - muat korpus, ekstraksi rujukan **v1** (logika asli) dan simpan artefak.
2. Tahap 1b - ekstraksi **v2** (perbaikan) + **validasi struktural otomatis** (seluruh korpus).
3. Tahap 1c - **audit akurasi**: sampling terstrata -> lembar `lembar_audit.xlsx` -> anotasi manual -> precision, kappa, recall.
4. Tahap 1d - bangun ekstraksi norma deontik.
5. Tahap 2 - bangun knowledge graph memakai ekstraksi yang dipilih (`EKSTRAKSI_AKTIF`) dan statistiknya.

**Cara menjalankan di Colab:** jalankan sel dari atas sampai sel pembuatan lembar audit (Tahap 1c-1). Buka `lembar_audit.xlsx` di folder Drive Anda (klik dua kali -> Google Spreadsheet, mode edit Office menyimpan balik ke .xlsx), isi kolom kuning, lalu jalankan sel evaluasi (Tahap 1c-2) dan seterusnya. Sel evaluasi aman dijalankan sebelum anotasi (hanya menampilkan petunjuk).

In [1]:
import os, re, glob, json, time, pickle, sys, random, math
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import networkx as nx

IN_COLAB = "google.colab" in sys.modules
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
pd.set_option('display.max_colwidth', 120)

print("Setup selesai | Colab:", IN_COLAB)
print(f"python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__} | networkx {nx.__version__}")

Setup selesai | Colab: True
python 3.13.18 | pandas 2.2.3 | numpy 2.1.3 | networkx 3.7


In [2]:
# ===================== KONFIGURASI (ubah hanya di sel ini) =====================
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/Project 7')   # sesuaikan
else:
    # Jupyter lokal: set variabel lingkungan PROJECT7_DIR atau ubah path di bawah
    BASE_DIR = Path(os.environ.get('PROJECT7_DIR', 'Project 7'))

KORPUS_DIR = BASE_DIR / 'output_parser'
IFACT_DIR  = BASE_DIR / 'artifacts'          # indeks, graf, embedding
AUDIT_DIR  = BASE_DIR / 'audit_rujukan'      # lembar audit & hasil evaluasi
for d in (KORPUS_DIR, IFACT_DIR, AUDIT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Ekstraksi yang dipakai membangun KG: 'v1' (asli) atau 'v2' (perbaikan).
# Ubah ke 'v2' HANYA setelah audit (Tahap 1c) mendukungnya.
EKSTRAKSI_AKTIF = 'v1'

# --- parameter audit akurasi ---
KONTEKS = 150                  # karakter konteks kiri/kanan pada lembar audit
N_INT_TAK_DITEMUKAN = 60       # internal: status "tak ditemukan"
N_INT_BEDA          = 60       # internal: valid tetapi hasil v1 != v2
N_INT_SAMA          = 80       # internal: valid dan v1 == v2
N_EKS_LUAR_SAMA     = 60       # eksternal luar-korpus, v1 == v2 (dalam-korpus & v1!=v2: diaudit semua)
N_EKS_V2_SAJA       = 40       # eksternal yang hanya ditemukan v2
N_PROBE_S1          = 40       # probe recall: chunk dengan >=1 rujukan terekstraksi
N_PROBE_S2          = 40       # probe recall: chunk bercue rujukan tetapi nol terekstraksi
BOOTSTRAP_B         = 2000
OVERWRITE_LEMBAR    = False    # True hanya jika ingin membuat ulang lembar (anotasi lama HILANG)
LEMBAR_AUDIT        = AUDIT_DIR / 'lembar_audit.xlsx'
TIMESTAMP           = datetime.now().strftime('%Y%m%d_%H%M')

print("Korpus dicari di:", KORPUS_DIR)
print("File terdeteksi :", sorted(os.path.basename(f) for f in glob.glob(os.path.join(KORPUS_DIR, 'korpus*.csv'))))

Mounted at /content/drive
Korpus dicari di: /content/drive/MyDrive/Project 7/output_parser
File terdeteksi : ['korpusPM172018.csv', 'korpusPM42025.csv', 'korpusPM52021.csv', 'korpusPM82024.csv', 'korpusPM82026.csv', 'korpusPM92023.csv', 'korpusPM_2_2023.csv', 'korpusPM_2_2025.csv', 'korpusPP462021.csv', 'korpusPP492005.csv', 'korpusPP512002.csv', 'korpusPP522000.csv', 'korpusPP532000.csv', 'korpusPP_28_2025.csv', 'korpusPerdirjen_3_2024.csv', 'korpusPerdirjen_7_2021.csv', 'korpusPerppu22022.csv', 'korpusPerpres_174_2024.csv', 'korpusUU122011.csv', 'korpusUU322002.csv', 'korpusUU361999.csv', 'korpusUU382009.csv', 'korpusUU62023.csv']


In [3]:
SCHEMA = ['doc_id','nama_dokumen','jenis','tahun','nomor','bab','nama_bab','bagian',
          'paragraf','pasal','ayat','huruf','angka','tipe','isi','referensi','kata_kunci','entitas']

def load_korpus(korpus_dir: str) -> pd.DataFrame:
    frames, laporan = [], []
    for path in sorted(glob.glob(os.path.join(korpus_dir, 'korpus*.csv'))):
        df = pd.read_csv(path)
        hilang = [c for c in SCHEMA if c not in df.columns]
        if hilang:
            print(f"[LEWATI] {os.path.basename(path)} — kolom hilang: {hilang}")
            continue
        df = df[SCHEMA].copy()
        df['sumber_file'] = os.path.basename(path)
        frames.append(df)
        laporan.append((os.path.basename(path), len(df), df.doc_id.iloc[0]))
    korpus = pd.concat(frames, ignore_index=True)
    print(pd.DataFrame(laporan, columns=['file','baris','doc_id']).to_string(index=False))
    return korpus

def bersihkan(korpus: pd.DataFrame) -> pd.DataFrame:
    df = korpus.copy()
    # 1. Normalisasi teks
    df['isi'] = (df['isi'].astype(str)
                   .str.replace(r'\s+', ' ', regex=True)
                   .str.strip())
    df = df[df['isi'].str.len() > 0].copy()
    # 2. Standarisasi metadata
    df['jenis'] = df['jenis'].astype(str).str.strip()
    df['tahun'] = pd.to_numeric(df['tahun'], errors='coerce').astype('Int64')
    for c in ['pasal','ayat','angka']:
        df[c] = (df[c].astype('string').str.strip()
                       .str.replace(r'\.0$', '', regex=True))   # '3.0' → '3'
    df['huruf'] = df['huruf'].astype('string').str.strip().str.lower()
    # 3. Buang duplikat persis
    n0 = len(df)
    df = df.drop_duplicates(subset=['doc_id','pasal','ayat','huruf','angka','isi'])
    if n0 - len(df): print(f"Duplikat dibuang: {n0-len(df)}")
    # 4. chunk_id unik & hirarkis, mis. UU_6_2023__P154A__a1__h-g__n4
    def buat_id(r):
        parts = [str(r['doc_id']), f"P{r['pasal']}" if pd.notna(r['pasal']) else "P-"]
        if pd.notna(r['ayat']):  parts.append(f"a{r['ayat']}")
        if pd.notna(r['huruf']): parts.append(f"h{r['huruf']}")
        if pd.notna(r['angka']): parts.append(f"n{r['angka']}")
        return "__".join(parts)
    df['chunk_id'] = df.apply(buat_id, axis=1)
    # jika masih ada bentrok id (unit sama terpecah beberapa baris), beri sufiks urut
    dup = df.duplicated('chunk_id', keep=False)
    df.loc[dup, 'chunk_id'] = (df.loc[dup, 'chunk_id'] + '__' +
                               df.loc[dup].groupby('chunk_id').cumcount().astype(str))
    df = df.reset_index(drop=True)
    return df

korpus = bersihkan(load_korpus(KORPUS_DIR))
print(f"\nTotal satuan hukum: {len(korpus)} | Dokumen: {korpus.doc_id.nunique()}")
korpus.head(3)

                      file  baris            doc_id
        korpusPM172018.csv    630    Permen_17_2018
         korpusPM42025.csv      5     Permen_4_2025
         korpusPM52021.csv   1463     Permen_5_2021
         korpusPM82024.csv    246     Permen_8_2024
         korpusPM82026.csv    418     Permen_8_2026
         korpusPM92023.csv    436     Permen_9_2023
       korpusPM_2_2023.csv    323     Permen_2_2023
       korpusPM_2_2025.csv    163     Permen_2_2025
        korpusPP462021.csv    611        PP_46_2021
        korpusPP492005.csv     21        PP_49_2005
        korpusPP512002.csv    318        PP_51_2005
        korpusPP522000.csv    417        PP_52_2000
        korpusPP532000.csv    177        PP_53_2000
      korpusPP_28_2025.csv   2562        PP_28_2025
korpusPerdirjen_3_2024.csv    781  Perdirjen_3_2024
korpusPerdirjen_7_2021.csv   1886 Perdirjen_07_2021
     korpusPerppu22022.csv   6607     Perppu_2_2022
korpusPerpres_174_2024.csv    123  Perpres_174_2024
        korp

,doc_id,nama_dokumen,jenis,tahun,nomor,bab,nama_bab,bagian,paragraf,pasal,ayat,huruf,angka,tipe,isi,referensi,kata_kunci,entitas,sumber_file,chunk_id
0,Permen_17_2018,Permen Nomor 17 Tahun 2018,Permen,2018,17,BAB I,Ketentuan Umum,NaN,NaN,1,<NA>,<NA>,<NA>,Definisi,Dalam Peraturan Menteri ini yang dimaksud dengan:,Pasal 1,menteri,Menteri,korpusPM172018.csv,Permen_17_2018__P1
1,Permen_17_2018,Permen Nomor 17 Tahun 2018,Permen,2018,17,BAB I,Ketentuan Umum,NaN,NaN,1,<NA>,<NA>,1,Ketentuan,Komunikasi Radio adalah telekomunikasi dengan mempergunakan gelombang radio.,Pasal 1 angka 1,"radio, komunikasi, telekomunikasi, mempergunakan, gelombang",NaN,korpusPM172018.csv,Permen_17_2018__P1__n1
2,Permen_17_2018,Permen Nomor 17 Tahun 2018,Permen,2018,17,BAB I,Ketentuan Umum,NaN,NaN,1,<NA>,<NA>,2,Ketentuan,"Kegiatan Amatir Radio adalah Komunikasi Radio mengenai ilmu pengetahuan, penyelidikan teknis dan informasi yang berk...",Pasal 1 angka 2,"radio, kegiatan, amatir, komunikasi, mengenai",NaN,korpusPM172018.csv,Permen_17_2018__P1__n2


In [4]:
# EKSTRAKSI v1 (logika asli, tidak diubah; hanya ditambah posisi match 'ms'/'me' untuk konteks audit)
POLA_INTERNAL  = re.compile(r'Pasal\s+(\d+[A-Z]*)(?:\s+ayat\s*\((\d+)\))?', re.I)
POLA_EKSTERNAL = re.compile(
    r'(Undang-Undang'
    r'|Peraturan +Pemerintah(?: +Pengganti +Undang-Undang)?'
    r'|Peraturan +Presiden'
    r'|(?:Peraturan +Menteri|Peraturan +Direktur +Jenderal|Keputusan +Menteri)'
    r'(?: +(?!Nomor[ ,])[A-Za-z&,]+)*)'
    r' +Nomor +([0-9]+) +Tahun +([12][0-9][0-9][0-9])', re.I)

JENIS_MAP = {'undang-undang':'UU', 'peraturan pemerintah':'PP',
             'peraturan pemerintah pengganti undang-undang':'Perppu',
             'peraturan presiden':'Perpres', 'peraturan menteri':'Permen'}

def ekstrak_rujukan(df: pd.DataFrame):
    internal, eksternal = [], []
    doc_lookup = {}   # (jenis, nomor, tahun) -> doc_id
    for _, r in df.drop_duplicates('doc_id').iterrows():
        doc_lookup[(str(r['jenis']).lower(), str(r['nomor']), str(r['tahun']))] = r['doc_id']
    pasal_ada = set(zip(df['doc_id'], df['pasal'].astype(str)))
    for _, r in df.iterrows():
        teks = r['isi']
        for m in POLA_INTERNAL.finditer(teks):
            p_tuju = m.group(1)
            if str(p_tuju) != str(r['pasal']):     # abaikan rujukan ke diri sendiri
                internal.append({'chunk_id': r['chunk_id'], 'doc_id': r['doc_id'],
                                 'pasal_asal': r['pasal'], 'pasal_tuju': p_tuju,
                                 'ayat_tuju': m.group(2),
                                 'valid': (r['doc_id'], p_tuju) in pasal_ada,
                                 'ms': m.start(), 'me': m.end()})
        for m in POLA_EKSTERNAL.finditer(teks):
            jenis_key = re.sub(r'\s+', ' ', m.group(1).lower())
            jenis = JENIS_MAP.get(jenis_key, m.group(1))
            key = (jenis.lower(), m.group(2), m.group(3))
            tuju = None
            for (j, n, t), did in doc_lookup.items():
                if n == m.group(2) and t == m.group(3):
                    tuju = did; break
            eksternal.append({'chunk_id': r['chunk_id'], 'doc_id': r['doc_id'],
                              'jenis_tuju': jenis, 'nomor': m.group(2), 'tahun': m.group(3),
                              'doc_tuju': tuju, 'dalam_korpus': tuju is not None,
                              'ms': m.start(), 'me': m.end()})
    return pd.DataFrame(internal), pd.DataFrame(eksternal)

ref_internal, ref_eksternal = ekstrak_rujukan(korpus)

print("=== Rujukan internal antar-pasal ===")
print(f"Total: {len(ref_internal)} | valid (pasal tujuan ada): {ref_internal['valid'].sum()}"
      f" | tak ditemukan: {(~ref_internal['valid']).sum()}")
print("\n=== Rujukan antar-dokumen ===")
print(f"Total sebutan: {len(ref_eksternal)} | menuju dokumen dalam korpus: {ref_eksternal['dalam_korpus'].sum()}")
print("\nRegulasi luar korpus yang paling sering dirujuk (kandidat perluasan korpus):")
print(ref_eksternal[~ref_eksternal.dalam_korpus]
      .groupby(['jenis_tuju','nomor','tahun']).size().sort_values(ascending=False).head(10))

=== Rujukan internal antar-pasal ===
Total: 7196 | valid (pasal tujuan ada): 6799 | tak ditemukan: 397

=== Rujukan antar-dokumen ===
Total sebutan: 341 | menuju dokumen dalam korpus: 106

Regulasi luar korpus yang paling sering dirujuk (kandidat perluasan korpus):
jenis_tuju                                                                 nomor  tahun
UU                                                                         8      1981     23
                                                                           11     2020      8
                                                                           3      1989      7
PERATURAN DIREKTUR JENDERAL SUMBER DAYA DAN PERANGKAT POS DAN INFORMATIKA  07     2021      6
UU                                                                         36     2000      6
                                                                           7      1992      6
Perppu                                                                     1      

In [5]:
korpus['nomor'] = pd.to_numeric(korpus['nomor'], errors='coerce').astype('Int64') # menunjukkan bahwa kolom nomor dalam DataFrame korpus memiliki data yang tidak dapat dikonversi ke tipe integer saat menyimpan ke format Parquet.

korpus.to_parquet(os.path.join(IFACT_DIR, 'korpus_bersih.parquet'))
ref_internal.to_csv(os.path.join(IFACT_DIR, 'ref_internal.csv'), index=False)
ref_eksternal.to_csv(os.path.join(IFACT_DIR, 'ref_eksternal.csv'), index=False)
print("Artefak Tahap 1 tersimpan di", IFACT_DIR)

Artefak Tahap 1 tersimpan di /content/drive/MyDrive/Project 7/artifacts


## Tahap 1b - Ekstraksi v2 dan validasi struktural otomatis

Kelemahan ekstraksi v1 yang diperbaiki di v2 (semuanya **hipotesis** yang dinilai oleh audit Tahap 1c, bukan diasumsikan benar):

1. **"Pasal N <peraturan lain>" tercatat sebagai rujukan internal.** v1 menangkap setiap "Pasal N" tanpa melihat apakah pasal itu milik peraturan lain ("Pasal 5 Undang-Undang Nomor 36 Tahun 1999"). v2 memeriksa ekor kalimat dan memisahkannya sebagai rujukan eksternal.
2. **Daftar dan rentang pasal.** v1 melewatkan "Pasal 5 dan 6" (angka tanpa kata *Pasal*) dan tidak mengekspansi "Pasal 6 sampai dengan Pasal 8" (pasal tengah hilang). v2 mengekspansi keduanya sesuai pasal yang benar-benar ada.
3. **Pencocokan antardokumen.** v1 membandingkan nomor sebagai *string* ("07" != "7") dan mengabaikan jenis peraturan, sehingga PP 5/2021 bisa tertukar dengan Perdirjen 5/2021. v2 mencocokkan `(jenis, nomor-int, tahun)`.

**Validasi struktural** (otomatis, seluruh korpus) hanya membuktikan bahwa pasal/ayat/dokumen tujuan *ada*. Ini **bukan** akurasi: rujukan bisa "valid" secara struktural tetapi salah makna. Akurasi sesungguhnya diukur di Tahap 1c.

In [6]:
# ---------------- Ekstraksi v2 ----------------
ref_internal_v1, ref_eksternal_v1 = ref_internal.copy(), ref_eksternal.copy()   # baseline v1

JENIS_ALT = (r'(?P<jenis>Kitab\s+Undang-Undang\s+Hukum\s+\w+'
             r'|Undang-Undang\s+Dasar(?:\s+Negara\s+Republik\s+Indonesia\s+Tahun\s+1945)?'
             r'|Undang-Undang'
             r'|Peraturan\s+Pemerintah\s+Pengganti\s+Undang-Undang'
             r'|Peraturan\s+Pemerintah|Peraturan\s+Presiden|Peraturan\s+Menteri'
             r'|Peraturan\s+Direktur\s+Jenderal|Keputusan\s+Menteri'
             r'|Keputusan\s+Direktur\s+Jenderal)')
SISA = r'(?P<sisa>(?:\s+(?!Nomor\b|ini\b)[A-Za-z&,]+){0,10}?)'
EKS_PENUH = re.compile(JENIS_ALT + SISA + r'\s+Nomor\s+(?P<nomor>\d+)\w*(?:/[\w./-]+)?\s+Tahun\s+(?P<tahun>\d{4})')
EKS_EKOR_PENUH = re.compile(r'\s*,?\s*(?:(?:pada|dalam|dari|yang\s+diatur\s+dalam)\s+)?' + EKS_PENUH.pattern)
EKS_EKOR_JENIS = re.compile(r'\s*,?\s*(?:(?:pada|dalam|dari|yang\s+diatur\s+dalam)\s+)?' + JENIS_ALT)

ANCHOR = re.compile(r'\b[Pp]asal\s+(\d+[A-Z]{0,2})(?![A-Za-z\d])')
QUAL = re.compile(
    r'(?:\s+ayat\s*\(\d+[A-Za-z]?\)(?:\s*(?:,\s*(?:dan|atau|serta)|,|dan|atau|serta)\s*(?:ayat\s*)?\(\d+[A-Za-z]?\))*'
    r'|\s+huruf\s+[a-z](?![a-z])(?:\s*(?:,\s*(?:dan|atau|serta)|,|dan|atau|serta)\s*[a-z](?![a-z]))*'
    r'|\s+angka\s+\d+)*')
SAMBUNG = re.compile(
    r'(?P<kon>\s*(?:,\s*(?:dan|serta|atau)|,|dan|serta|atau)\s*|\s+(?:sampai\s+dengan|s\.d\.|sd)\s+)'
    r'(?:[Pp]asal\s+(?P<p1>\d+[A-Z]{0,2})(?![A-Za-z\d])'
    r'|(?P<p2>\d+[A-Z]{0,2})(?![A-Za-z\d])(?!\s*\))'
    r'(?!\s+(?:hari|bulan|tahun|minggu|jam|persen|kali|orang|unit|kHz|MHz|GHz|Hz|dB|%)))')
AYAT_DI = re.compile(r'ayat\s*\((\d+[A-Za-z]?)\)')
KODE_DOCID = {'UU': 'UU', 'PP': 'PP', 'PERPPU': 'PERPPU', 'PERPRES': 'PERPRES',
              'PERMEN': 'PERMEN', 'PERDIRJEN': 'PERDIRJEN', 'KEPMEN': 'KEPMEN'}

def kode_dari_teks(t):
    t = re.sub(r'\s+', ' ', t.lower())
    for awal, kode in [('peraturan pemerintah pengganti', 'PERPPU'), ('peraturan pemerintah', 'PP'),
                       ('peraturan direktur', 'PERDIRJEN'), ('peraturan presiden', 'PERPRES'),
                       ('peraturan menteri', 'PERMEN'), ('keputusan menteri', 'KEPMEN'),
                       ('keputusan direktur', 'KEPDIRJEN'), ('undang-undang dasar', 'UUD'),
                       ('undang-undang', 'UU'), ('kitab', 'KUH')]:
        if t.startswith(awal):
            return kode
    return 'LAIN'

def parse_docid(doc_id):
    m = re.match(r'([A-Za-z]+)_0*(\d+)_(\d{4})', str(doc_id))
    return None if not m else (KODE_DOCID.get(m.group(1).upper(), m.group(1).upper()), int(m.group(2)), int(m.group(3)))

def basis_pasal(p):
    return int(re.match(r'\d+', p).group())

COL_V2I = ['chunk_id', 'doc_id', 'pasal_asal', 'pasal_tuju', 'ayat_tuju', 'sifat', 'doc_tuju',
           'pola', 'valid', 'ayat_valid', 'ms', 'me']
COL_V2E = ['chunk_id', 'doc_id', 'kode_jenis', 'nomor', 'tahun', 'doc_tuju', 'dalam_korpus',
           'ambigu_jenis', 'rujukan_diri', 'ms', 'me']

def ekstrak_v2(df):
    pasal_per_doc, ayat_per_pasal, lookup, asli = {}, {}, {}, {}
    for did, g in df.groupby('doc_id'):
        pasal_per_doc[did] = {str(p).upper() for p in g['pasal'].dropna()}
        k = parse_docid(did)
        if k:
            lookup[k] = did
        for p in g['pasal'].dropna():
            asli[(did, str(p).upper())] = str(p)
    for (did, p), g in df.groupby(['doc_id', 'pasal']):
        ayat_per_pasal[(did, str(p).upper())] = {str(a).upper() for a in g['ayat'].dropna()}
    internal, eksternal = [], []
    for _, r in df.iterrows():
        teks, did = r['isi'], r['doc_id']
        asal = str(r['pasal']).upper()
        anchors = list(ANCHOR.finditer(teks))
        i = 0
        while i < len(anchors):
            a = anchors[i]
            items = [(a.group(1).upper(), False)]
            akhir = QUAL.match(teks, a.end()).end()
            ayat_list = AYAT_DI.findall(teks[a.end():akhir])
            while True:
                m2 = SAMBUNG.match(teks, akhir)
                if not m2:
                    break
                num = (m2.group('p1') or m2.group('p2')).upper()
                kon = m2.group('kon')
                items.append((num, 'sampai' in kon or 's.d.' in kon or 'sd' in kon.split()))
                akhir = QUAL.match(teks, m2.end()).end()
                ayat_list += AYAT_DI.findall(teks[m2.end():akhir])
            i += 1
            while i < len(anchors) and anchors[i].start() < akhir:
                i += 1
            sifat, doc_tuju = 'internal', did
            mfull = EKS_EKOR_PENUH.match(teks, akhir)
            if mfull:
                k = (kode_dari_teks(mfull.group('jenis')), int(mfull.group('nomor')), int(mfull.group('tahun')))
                sifat, doc_tuju = 'eksternal_bernomor', lookup.get(k)
            else:
                mj = EKS_EKOR_JENIS.match(teks, akhir)
                if mj and not re.match(r'\s+ini\b', teks[mj.end('jenis'):]):
                    sifat, doc_tuju = 'eksternal_tanpa_nomor', None
            target, pola = [], 'tunggal'
            for idx, (num, rentang) in enumerate(items):
                if rentang and idx > 0:
                    pola = 'rentang'
                    lo, hi = basis_pasal(items[idx - 1][0]), basis_pasal(num)
                    if doc_tuju and doc_tuju in pasal_per_doc and 0 <= hi - lo <= 60:
                        target += sorted(p for p in pasal_per_doc[doc_tuju] if lo <= basis_pasal(p) <= hi)
                    else:
                        target.append(num)
                else:
                    if len(items) > 1 and pola == 'tunggal':
                        pola = 'daftar'
                    target.append(num)
            for p in dict.fromkeys(target):
                if sifat == 'internal' and p == asal:
                    continue
                valid = (p in pasal_per_doc[doc_tuju]) if (doc_tuju and doc_tuju in pasal_per_doc) else None
                ayat_valid = None
                if valid and len(set(target)) == 1 and ayat_list:
                    ada = ayat_per_pasal.get((doc_tuju, p), set())
                    ayat_valid = all(x.upper() in ada for x in ayat_list) if ada else None
                internal.append({'chunk_id': r['chunk_id'], 'doc_id': did, 'pasal_asal': r['pasal'],
                                 'pasal_tuju': asli.get((doc_tuju, p), p), 'ayat_tuju': ','.join(ayat_list) or None,
                                 'sifat': sifat, 'doc_tuju': doc_tuju, 'pola': pola, 'valid': valid,
                                 'ayat_valid': ayat_valid, 'ms': a.start(), 'me': akhir})
        for m in EKS_PENUH.finditer(teks):
            k = (kode_dari_teks(m.group('jenis')), int(m.group('nomor')), int(m.group('tahun')))
            tuju = lookup.get(k)
            ambigu = tuju is None and any(kk[1:] == k[1:] for kk in lookup)
            eksternal.append({'chunk_id': r['chunk_id'], 'doc_id': did, 'kode_jenis': k[0], 'nomor': k[1],
                              'tahun': k[2], 'doc_tuju': tuju, 'dalam_korpus': tuju is not None,
                              'ambigu_jenis': ambigu, 'rujukan_diri': tuju == did,
                              'ms': m.start(), 'me': m.end()})
    return pd.DataFrame(internal, columns=COL_V2I), pd.DataFrame(eksternal, columns=COL_V2E)

ref_internal_v2, ref_eksternal_v2 = ekstrak_v2(korpus)

def n_valid(s):
    return int((s == True).sum())   # noqa: E712

perbandingan = pd.DataFrame([
    ['Rujukan internal - total', len(ref_internal_v1), len(ref_internal_v2)],
    ['  valid struktural (pasal tujuan ada)', n_valid(ref_internal_v1['valid']), n_valid(ref_internal_v2['valid'])],
    ['  dari v2: dikenali sebagai rujukan eksternal', '-', int(ref_internal_v2['sifat'].str.startswith('eksternal').sum())],
    ['  dari v2: hasil ekspansi daftar/rentang', '-', int((ref_internal_v2['pola'] != 'tunggal').sum())],
    ['Rujukan antardokumen - total sebutan', len(ref_eksternal_v1), len(ref_eksternal_v2)],
    ['  menuju dokumen dalam korpus', int(ref_eksternal_v1['dalam_korpus'].sum()) if len(ref_eksternal_v1) else 0,
     int(ref_eksternal_v2['dalam_korpus'].sum())],
    ['  v2: sebutan jenis-ambigu (nomor/tahun sama, jenis beda)', '-', int(ref_eksternal_v2['ambigu_jenis'].sum())],
], columns=['Ukuran', 'v1 (asli)', 'v2 (perbaikan)'])
display(perbandingan)

,Ukuran,v1 (asli),v2 (perbaikan)
0,Rujukan internal - total,7196,7111
1,valid struktural (pasal tujuan ada),6799,6702
2,dari v2: dikenali sebagai rujukan eksternal,-,178
3,dari v2: hasil ekspansi daftar/rentang,-,1946
4,Rujukan antardokumen - total sebutan,341,304
5,menuju dokumen dalam korpus,106,46
6,"v2: sebutan jenis-ambigu (nomor/tahun sama, jenis beda)",-,29


In [7]:
# ---------------- Validasi struktural otomatis (seluruh korpus) ----------------
pasal_maks = (korpus.assign(pb=pd.to_numeric(korpus['pasal'].astype('string').str.extract(r'(\d+)')[0], errors='coerce'))
                    .groupby('doc_id')['pb'].max())

def tabel_internal(v, nama):
    if len(v) == 0:
        return pd.DataFrame()
    t = (v.assign(ok=(v['valid'] == True), gagal=(v['valid'] == False), nacek=v['valid'].isna())   # noqa: E712
          .groupby('doc_id').agg(total=('ok', 'size'), valid=('ok', 'sum'), tak_ditemukan=('gagal', 'sum'),
                                 tak_dapat_dicek=('nacek', 'sum')))
    t['%valid_dari_yang_dicek'] = (100 * t['valid'] / (t['valid'] + t['tak_ditemukan']).replace(0, np.nan)).round(1)
    t.columns = [f'{nama}_{c}' for c in t.columns]
    return t

tab_int = tabel_internal(ref_internal_v1, 'v1').join(tabel_internal(ref_internal_v2, 'v2'), how='outer')
print("=== Validasi struktural rujukan INTERNAL per dokumen ===")
print("(tak_dapat_dicek = pasal milik peraturan lain di luar korpus; hanya ada pada v2)")
display(tab_int)

# Diagnosis cepat rujukan "tak ditemukan" pada v1: pasal tujuan melebihi pasal terbesar dokumen -> kemungkinan
# rujukan ke peraturan lain (bukan galat parser pasal tujuan).
tak = ref_internal_v1[~ref_internal_v1['valid']].copy() if len(ref_internal_v1) else ref_internal_v1
if len(tak):
    tak['pb_tuju'] = tak['pasal_tuju'].str.extract(r'(\d+)')[0].astype(float)
    tak['melebihi_pasal_maks'] = tak['pb_tuju'] > tak['doc_id'].map(pasal_maks)
    print(f"\nv1 'tak ditemukan': {len(tak)} | di antaranya pasal tujuan melebihi pasal maksimum dokumen "
          f"(indikasi rujukan ke peraturan lain): {int(tak['melebihi_pasal_maks'].sum())}")

print("\n=== Validasi struktural rujukan ANTARDOKUMEN ===")
e1 = ref_eksternal_v1.groupby('doc_id').agg(v1_sebutan=('doc_id', 'size'), v1_dalam_korpus=('dalam_korpus', 'sum')) if len(ref_eksternal_v1) else pd.DataFrame()
e2 = ref_eksternal_v2.groupby('doc_id').agg(v2_sebutan=('doc_id', 'size'), v2_dalam_korpus=('dalam_korpus', 'sum'),
                                             v2_jenis_ambigu=('ambigu_jenis', 'sum'), v2_rujukan_diri=('rujukan_diri', 'sum')) if len(ref_eksternal_v2) else pd.DataFrame()
display(e1.join(e2, how='outer') if len(e1) or len(e2) else "tidak ada sebutan peraturan lain")

if len(ref_internal_v2):
    print("\nv2 - ayat tujuan tidak ada pada pasal tujuan:", int((ref_internal_v2['ayat_valid'] == False).sum()))  # noqa: E712
    print("v2 - sebaran sifat rujukan:", ref_internal_v2['sifat'].value_counts().to_dict())

ref_internal_v2.to_csv(AUDIT_DIR / f'ref_internal_v2_{TIMESTAMP}.csv', index=False, encoding='utf-8-sig')
ref_eksternal_v2.to_csv(AUDIT_DIR / f'ref_eksternal_v2_{TIMESTAMP}.csv', index=False, encoding='utf-8-sig')
tab_int.to_csv(AUDIT_DIR / f'validasi_struktural_internal_{TIMESTAMP}.csv', encoding='utf-8-sig')
print("\nTersimpan di", AUDIT_DIR)

=== Validasi struktural rujukan INTERNAL per dokumen ===
(tak_dapat_dicek = pasal milik peraturan lain di luar korpus; hanya ada pada v2)


,v1_total,v1_valid,v1_tak_ditemukan,v1_tak_dapat_dicek,v1_%valid_dari_yang_dicek,v2_total,v2_valid,v2_tak_ditemukan,v2_tak_dapat_dicek,v2_%valid_dari_yang_dicek
doc_id,,,,,,,,,,
PP_28_2025,570,482,88,0,84.6,572,531,41,0,92.8
PP_46_2021,137,125,12,0,91.2,139,118,3,18,97.5
PP_49_2005,6,5,1,0,83.3,6,4,1,1,80.0
PP_51_2005,57,57,0,0,100.0,57,56,0,1,100.0
PP_52_2000,80,80,0,0,100.0,82,82,0,0,100.0
PP_53_2000,21,20,1,0,95.2,24,22,2,0,91.7
Perdirjen_07_2021,351,272,79,0,77.5,351,205,43,103,82.7
Perdirjen_3_2024,100,100,0,0,100.0,100,100,0,0,100.0
Permen_17_2018,98,98,0,0,100.0,104,104,0,0,100.0



v1 'tak ditemukan': 397 | di antaranya pasal tujuan melebihi pasal maksimum dokumen (indikasi rujukan ke peraturan lain): 83

=== Validasi struktural rujukan ANTARDOKUMEN ===


,v1_sebutan,v1_dalam_korpus,v2_sebutan,v2_dalam_korpus,v2_jenis_ambigu,v2_rujukan_diri
doc_id,,,,,,
PP_28_2025,9,9,7.0,7.0,0.0,0.0
PP_46_2021,6,3,5.0,2.0,0.0,0.0
PP_49_2005,3,3,2.0,2.0,0.0,0.0
PP_51_2005,5,5,4.0,4.0,0.0,0.0
PP_52_2000,10,1,9.0,0.0,0.0,0.0
Perdirjen_07_2021,81,46,69.0,17.0,22.0,0.0
Perdirjen_3_2024,3,3,1.0,1.0,0.0,0.0
Permen_17_2018,4,3,1.0,0.0,0.0,0.0
Permen_2_2023,4,2,2.0,0.0,0.0,0.0



v2 - ayat tujuan tidak ada pada pasal tujuan: 74
v2 - sebaran sifat rujukan: {'internal': 6933, 'eksternal_bernomor': 138, 'eksternal_tanpa_nomor': 40}

Tersimpan di /content/drive/MyDrive/Project 7/audit_rujukan


## Tahap 1c - Audit akurasi rujukan

**Mengapa audit manual?** Tidak ada data kebenaran (*ground truth*) rujukan pada korpus; akurasi hanya bisa diklaim dari pemeriksaan manusia pada sampel. Protokol:

- **Sampling terstrata** (seed tetap, dapat direproduksi). Strata berisiko tinggi (rujukan "tak ditemukan", hasil v1 != v2) diambil lebih banyak, lalu hasil **dibobot ulang** ke ukuran populasi (`N_strata`) saat menghitung precision.
- **Dua anotator independen** mengisi `label_A` dan `label_B`; selisih diselesaikan di `label_adjudikasi`. Bila hanya satu anotator, isi `label_A` saja (evaluasi tetap jalan, kappa tidak dilaporkan, dan keterbatasan ini harus ditulis di metodologi).
- **Label** (menilai `hasil_v1`): `BENAR` | `SALAH_TARGET` (jenis benar, pasal/dokumen salah) | `SALAH_JENIS` (mis. pasal peraturan lain dibaca internal) | `BUKAN_RUJUKAN`. Bila bukan BENAR, isi `kelas_benar` (INTERNAL / EKSTERNAL / BUKAN_RUJUKAN).
- **Probe recall** (sheet `probe`): hitung rujukan sebenarnya pada sampel chunk, lalu berapa yang tertangkap v1 dan v2. Rujukan antar-ayat dalam satu pasal tidak dihitung.

**Metrik:** precision berbobot strata + CI95 bootstrap, precision per strata + CI Wilson, Cohen's kappa, diagnosis galat, recall v1 dan v2.

### Tahap 1c-2 - Evaluasi akurasi (jalankan setelah lembar diisi)

Precision = proporsi `BENAR` pada label final (adjudikasi, atau label A bila A == B / hanya satu anotator). Estimasi populasi = jumlah bobot `N_strata/N` x precision tiap strata; CI95 dari *bootstrap* terstrata (seed tetap). Recall dari probe memakai estimator rasio berbobot strata. Hasil dan tabel siap salin ke bab 4 disimpan ke folder audit dengan stempel waktu.

In [8]:
# ---------------- Sampling terstrata & lembar audit ----------------
LABEL_SET = ['BENAR', 'SALAH_TARGET', 'SALAH_JENIS', 'BUKAN_RUJUKAN']
KELAS_SET = ['INTERNAL', 'EKSTERNAL', 'BUKAN_RUJUKAN']
CUE = re.compile(r'\b[Pp]asal\s+\d+|Nomor\s+\d+\s+Tahun\s+\d{4}|Undang-Undang|'
                 r'Peraturan\s+(?:Pemerintah|Presiden|Menteri|Direktur)|Keputusan\s+Menteri')

def konteks(teks, s, e, w=KONTEKS):
    return teks[max(0, s - w):s] + '【' + teks[s:e] + '】' + teks[e:e + w]

def ambil(d, n, rng):
    return d.copy() if len(d) <= n else d.iloc[np.sort(rng.choice(len(d), size=n, replace=False))].copy()

def ringkas_v2(rows):
    if rows is None or len(rows) == 0:
        return '(tidak ada)'
    out = []
    for _, x in rows.iterrows():
        tujuan = f"{x['doc_tuju']}::P{x['pasal_tuju']}" if x['doc_tuju'] else f"[luar]::P{x['pasal_tuju']}"
        out.append(f"{x['sifat']}:{tujuan}" + (f" ayat {x['ayat_tuju']}" if x['ayat_tuju'] else ''))
    return ' | '.join(out)

def stratifikasi(pop, kuota, rng):
    out = []
    for s, g in pop.groupby('strata'):
        samp = ambil(g, kuota[s], rng)
        samp['N_strata'], samp['n_strata'] = len(g), len(samp)
        out.append(samp)
    return pd.concat(out, ignore_index=True) if out else pop

def sampel_internal(df, v1i, v2i, rng):
    teks_by = df.set_index('chunk_id')['isi']
    v2_by = {c: g for c, g in v2i.groupby('chunk_id')} if len(v2i) else {}
    rows = []
    for _, x in v1i.iterrows():
        g = v2_by.get(x['chunk_id'], pd.DataFrame(columns=COL_V2I))
        semua = g[(g['ms'] <= x['ms']) & (g['me'] >= x['me'])]
        pad = semua[semua['pasal_tuju'].astype(str).str.upper() == str(x['pasal_tuju']).upper()]
        if len(pad) == 0:
            pad = semua
        sama = len(pad) > 0 and pad.iloc[0]['sifat'] == 'internal' and \
            str(pad.iloc[0]['pasal_tuju']).upper() == str(x['pasal_tuju']).upper()
        strata = 'int_tak_ditemukan' if not x['valid'] else ('int_valid_sama' if sama else 'int_valid_v1v2_beda')
        rows.append({'jenis_rujukan': 'internal', 'strata': strata, 'doc_id': x['doc_id'], 'chunk_id': x['chunk_id'],
                     'pasal_asal': x['pasal_asal'], 'konteks': konteks(teks_by[x['chunk_id']], x['ms'], x['me']),
                     'hasil_v1': f"internal:{x['doc_id']}::P{x['pasal_tuju']}" + (f" ayat {x['ayat_tuju']}" if x['ayat_tuju'] else ''),
                     'status_sistem': 'valid' if x['valid'] else 'tak_ditemukan', 'hasil_v2': ringkas_v2(pad),
                     'v2_kelas': 'TIDAK_DITANGKAP' if len(pad) == 0 else pad.iloc[0]['sifat'].upper()})
    pop = pd.DataFrame(rows)
    kuota = {'int_tak_ditemukan': N_INT_TAK_DITEMUKAN, 'int_valid_v1v2_beda': N_INT_BEDA, 'int_valid_sama': N_INT_SAMA}
    return stratifikasi(pop, kuota, rng) if len(pop) else pop

def sampel_eksternal(df, v1e, v2e, rng):
    teks_by = df.set_index('chunk_id')['isi']
    v2_by = {c: g for c, g in v2e.groupby('chunk_id')} if len(v2e) else {}
    rows = []
    for _, x in v1e.iterrows():
        g = v2_by.get(x['chunk_id'], pd.DataFrame(columns=COL_V2E))
        pad = g[(g['ms'] < x['me']) & (g['me'] > x['ms'])]
        v2_ada = len(pad) > 0
        v2_doc = pad.iloc[0]['doc_tuju'] if v2_ada else None
        sama = v2_ada and (v2_doc == x['doc_tuju'] or (pd.isna(v2_doc) and pd.isna(x['doc_tuju'])))
        strata = 'eks_dalam_korpus' if x['dalam_korpus'] else ('eks_luar_sama' if sama else 'eks_luar_v1v2_beda')
        rows.append({'jenis_rujukan': 'eksternal', 'strata': strata, 'doc_id': x['doc_id'], 'chunk_id': x['chunk_id'],
                     'pasal_asal': '', 'konteks': konteks(teks_by[x['chunk_id']], x['ms'], x['me']),
                     'hasil_v1': f"{x['jenis_tuju']} {x['nomor']}/{x['tahun']} -> {x['doc_tuju'] if x['dalam_korpus'] else '[luar korpus]'}",
                     'status_sistem': 'dalam_korpus' if x['dalam_korpus'] else 'luar_korpus',
                     'hasil_v2': (f"{pad.iloc[0]['kode_jenis']} {pad.iloc[0]['nomor']}/{pad.iloc[0]['tahun']} -> "
                                  f"{v2_doc if pd.notna(v2_doc) else '[luar korpus]'}"
                                  + (' (jenis-ambigu)' if pad.iloc[0]['ambigu_jenis'] else '')) if v2_ada else 'TIDAK_DITANGKAP',
                     'v2_kelas': 'SAMA' if sama else 'BEDA'})
    for _, y in v2e.iterrows():
        tertangkap = len(v1e) > 0 and bool(((v1e['chunk_id'] == y['chunk_id']) & (v1e['ms'] < y['me'])
                                            & (v1e['me'] > y['ms'])).any())
        if not tertangkap:
            rows.append({'jenis_rujukan': 'eksternal', 'strata': 'eks_v2_saja', 'doc_id': y['doc_id'], 'chunk_id': y['chunk_id'],
                         'pasal_asal': '', 'konteks': konteks(teks_by[y['chunk_id']], y['ms'], y['me']),
                         'hasil_v1': 'TIDAK_DITANGKAP', 'status_sistem': 'tidak_ada',
                         'hasil_v2': f"{y['kode_jenis']} {y['nomor']}/{y['tahun']} -> {y['doc_tuju'] if pd.notna(y['doc_tuju']) else '[luar korpus]'}",
                         'v2_kelas': 'V2_SAJA'})
    pop = pd.DataFrame(rows)
    kuota = {'eks_dalam_korpus': 10**9, 'eks_luar_v1v2_beda': 10**9, 'eks_luar_sama': N_EKS_LUAR_SAMA, 'eks_v2_saja': N_EKS_V2_SAJA}
    return stratifikasi(pop, kuota, rng) if len(pop) else pop

def sampel_probe(df, v1i, v1e, v2i, v2e, rng):
    ada = set(v1i['chunk_id']) | set(v1e['chunk_id']) | set(v2i['chunk_id']) | set(v2e['chunk_id'])
    bercue = df[df['isi'].apply(lambda t: bool(CUE.search(t)))]
    s1, s2 = bercue[bercue['chunk_id'].isin(ada)], bercue[~bercue['chunk_id'].isin(ada)]
    def v1_list(c):
        a, b = v1i[v1i['chunk_id'] == c], v1e[v1e['chunk_id'] == c]
        h = [f"int:P{x['pasal_tuju']}" for _, x in a.iterrows()] + [f"eks:{x['jenis_tuju']} {x['nomor']}/{x['tahun']}" for _, x in b.iterrows()]
        return ' | '.join(h) if h else '(kosong)'
    def v2_list(c):
        a, b = v2i[v2i['chunk_id'] == c], v2e[v2e['chunk_id'] == c]
        h = [f"{x['sifat']}:P{x['pasal_tuju']}" for _, x in a.iterrows()] + [f"eks:{x['kode_jenis']} {x['nomor']}/{x['tahun']}" for _, x in b.iterrows()]
        return ' | '.join(h) if h else '(kosong)'
    out = []
    for nama, g, n in (('probe_ada_ekstraksi', s1, N_PROBE_S1), ('probe_tanpa_ekstraksi', s2, N_PROBE_S2)):
        samp = ambil(g, n, rng)
        out.append(pd.DataFrame({'jenis_rujukan': 'probe', 'strata': nama, 'doc_id': samp['doc_id'].values,
                                 'chunk_id': samp['chunk_id'].values, 'teks_lengkap': samp['isi'].values,
                                 'hasil_v1': [v1_list(c) for c in samp['chunk_id']],
                                 'hasil_v2': [v2_list(c) for c in samp['chunk_id']],
                                 'N_strata': len(g), 'n_strata': len(samp)}))
    return pd.concat(out, ignore_index=True)

ILLEGAL = re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f]')

def tulis_lembar(path, s_int, s_eks, s_probe):
    from openpyxl import Workbook
    from openpyxl.styles import Alignment, Font, PatternFill
    from openpyxl.worksheet.datavalidation import DataValidation
    wb = Workbook()
    f_h, fill_h = Font(name='Arial', bold=True, color='FFFFFF'), PatternFill('solid', start_color='305496')
    fill_in, f_n = PatternFill('solid', start_color='FFF2CC'), Font(name='Arial', size=10)
    ws0 = wb.active
    ws0.title = 'PETUNJUK'
    for r in [['PETUNJUK ANOTASI (kolom kuning = diisi anotator)'], [''],
              ['Anotator mengisi label_A / label_B SECARA INDEPENDEN; selisih diputuskan di label_adjudikasi. Satu anotator: isi label_A saja.'],
              [''], ['BENAR', 'hasil_v1 tepat: kelas (internal/eksternal) dan target (pasal/dokumen) sesuai teks.'],
              ['SALAH_TARGET', 'Memang rujukan dan kelasnya benar, tetapi pasal/dokumen target keliru atau tidak lengkap.'],
              ['SALAH_JENIS', 'Kelas keliru, mis. "Pasal 5 Undang-Undang No. X" dibaca sebagai rujukan internal.'],
              ['BUKAN_RUJUKAN', 'Bukan rujukan normatif (sebutan judul, kutipan bunyi pasal yang diubah, angka lain).'],
              ['kelas_benar', 'Wajib diisi bila label bukan BENAR: INTERNAL / EKSTERNAL / BUKAN_RUJUKAN.'],
              ['status tak_ditemukan', 'BENAR = rujukan internal yang sah walau pasal tujuan tidak ada di korpus.'],
              ['eksternal', 'BENAR = dokumen tujuan (atau "luar korpus") sudah tepat.'], [''],
              ['SHEET probe', 'benar_total = jumlah rujukan normatif sebenarnya (tingkat pasal atau dokumen) pada teks_lengkap.'],
              ['', 'tertangkap_v1 / tertangkap_v2 = berapa dari rujukan benar itu tercantum pada hasil_v1 / hasil_v2.'],
              ['', 'Rujukan antar-ayat dalam satu pasal tidak dihitung. Satu anotator: isi kolom _A saja.'],
              [''], ['Kolom N_strata / n_strata dipakai untuk pembobotan - JANGAN diubah atau dihapus.']]:
        ws0.append(r)
    ws0['A1'].font = Font(name='Arial', bold=True, size=12)
    ws0.column_dimensions['A'].width = 24
    ws0.column_dimensions['B'].width = 110

    def sheet(nama, d, kolom, inp, dd, lebar):
        ws = wb.create_sheet(nama)
        ws.append(kolom)
        for c in ws[1]:
            c.font, c.fill, c.alignment = f_h, fill_h, Alignment(wrap_text=True, vertical='center')
        d = d.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
        for i, (_, r) in enumerate(d.iterrows(), start=1):
            vals = []
            for k in kolom:
                v = i if k == 'id' else (r[k] if k in d.columns and pd.notna(r[k]) else '')
                if isinstance(v, str):
                    v = ILLEGAL.sub('', v)[:30000]
                vals.append(v)
            ws.append(vals)
        for row in ws.iter_rows(min_row=2):
            for c in row:
                c.font, c.alignment = f_n, Alignment(wrap_text=True, vertical='top')
                if kolom[c.column - 1] in inp:
                    c.fill = fill_in
        for j, k in enumerate(kolom, start=1):
            ws.column_dimensions[ws.cell(1, j).column_letter].width = lebar.get(k, 16)
        for k, daftar in dd.items():
            dv = DataValidation(type='list', formula1='"' + ','.join(daftar) + '"', allow_blank=True)
            ws.add_data_validation(dv)
            col = ws.cell(1, kolom.index(k) + 1).column_letter
            dv.add(f'{col}2:{col}{max(ws.max_row, 2)}')
        ws.freeze_panes = 'A2'

    kol = ['id', 'jenis_rujukan', 'strata', 'doc_id', 'chunk_id', 'pasal_asal', 'konteks', 'hasil_v1', 'status_sistem',
           'hasil_v2', 'v2_kelas', 'label_A', 'label_B', 'label_adjudikasi', 'kelas_benar', 'catatan', 'N_strata', 'n_strata']
    lebar = {'konteks': 70, 'hasil_v1': 34, 'hasil_v2': 34, 'chunk_id': 28, 'catatan': 30, 'teks_lengkap': 90, 'strata': 20}
    dd = {'label_A': LABEL_SET, 'label_B': LABEL_SET, 'label_adjudikasi': LABEL_SET, 'kelas_benar': KELAS_SET}
    inp = {'label_A', 'label_B', 'label_adjudikasi', 'kelas_benar', 'catatan'}
    sheet('internal', s_int, kol, inp, dd, lebar)
    sheet('eksternal', s_eks, kol, inp, dd, lebar)
    kp = ['id', 'strata', 'doc_id', 'chunk_id', 'teks_lengkap', 'hasil_v1', 'hasil_v2', 'benar_total_A', 'tertangkap_v1_A',
          'tertangkap_v2_A', 'benar_total_B', 'tertangkap_v1_B', 'tertangkap_v2_B', 'catatan', 'N_strata', 'n_strata']
    sheet('probe', s_probe, kp, set(kp[7:14]), {}, lebar)
    wb.save(path)

if LEMBAR_AUDIT.exists() and not OVERWRITE_LEMBAR:
    print(f"Lembar audit SUDAH ADA dan tidak ditimpa (agar anotasi aman): {LEMBAR_AUDIT}")
    print("Set OVERWRITE_LEMBAR = True di sel konfigurasi hanya bila ingin membuat ulang.")
else:
    rng = np.random.default_rng(RANDOM_SEED)
    s_int = sampel_internal(korpus, ref_internal_v1, ref_internal_v2, rng)
    s_eks = sampel_eksternal(korpus, ref_eksternal_v1, ref_eksternal_v2, rng)
    s_probe = sampel_probe(korpus, ref_internal_v1, ref_eksternal_v1, ref_internal_v2, ref_eksternal_v2, rng)
    tulis_lembar(LEMBAR_AUDIT, s_int, s_eks, s_probe)
    print(f"Lembar audit dibuat: {LEMBAR_AUDIT}")
    print(f"  internal : {len(s_int)} baris | strata:", s_int.groupby('strata')['n_strata'].first().to_dict() if len(s_int) else {})
    print(f"  eksternal: {len(s_eks)} baris | strata:", s_eks.groupby('strata')['n_strata'].first().to_dict() if len(s_eks) else {})
    print(f"  probe    : {len(s_probe)} baris")
    print("\nLANGKAH BERIKUTNYA: buka file di atas, isi kolom kuning, simpan, lalu jalankan sel evaluasi di bawah.")

Lembar audit dibuat: /content/drive/MyDrive/Project 7/audit_rujukan/lembar_audit.xlsx
  internal : 200 baris | strata: {'int_tak_ditemukan': 60, 'int_valid_sama': 80, 'int_valid_v1v2_beda': 60}
  eksternal: 200 baris | strata: {'eks_dalam_korpus': 106, 'eks_luar_sama': 60, 'eks_luar_v1v2_beda': 20, 'eks_v2_saja': 14}
  probe    : 80 baris

LANGKAH BERIKUTNYA: buka file di atas, isi kolom kuning, simpan, lalu jalankan sel evaluasi di bawah.


In [9]:
# ---------------- Evaluasi akurasi ----------------
def wilson(k, n, z=1.96):
    if n == 0:
        return (float('nan'), float('nan'))
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))

def cohen_kappa(a, b):
    a, b = list(a), list(b); n = len(a)
    if n == 0:
        return float('nan')
    po = sum(x == y for x, y in zip(a, b)) / n
    pe = sum((a.count(k) / n) * (b.count(k) / n) for k in set(a) | set(b))
    return float('nan') if pe == 1 else (po - pe) / (1 - pe)

def tafsir_kappa(k):
    if k != k:
        return 'n/a'
    return ('lemah' if k < 0.4 else 'sedang' if k < 0.6 else 'kuat' if k < 0.8 else 'sangat kuat') if k >= 0 else 'di bawah acak'

def label_final(d):
    adj = d['label_adjudikasi'].where(d['label_adjudikasi'] != '')
    ok = (d['label_A'] != '') & ((d['label_B'] == '') | (d['label_A'] == d['label_B']))
    return adj.fillna(d['label_A'].where(ok))

def presisi_berbobot(d, rng, B=BOOTSTRAP_B):
    g = {s: x['benar'].to_numpy(float) for s, x in d.groupby('strata')}
    N = {s: float(d.loc[d['strata'] == s, 'N_strata'].iloc[0]) for s in g}
    tot = sum(N.values())
    est = lambda smp: sum(N[s] / tot * smp[s].mean() for s in smp)   # noqa: E731
    boot = [est({s: rng.choice(v, size=len(v), replace=True) for s, v in g.items()}) for _ in range(B)]
    return est(g), float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))

def evaluasi_sheet(nama, d, rng):
    d = d.copy()
    for c in ['label_A', 'label_B', 'label_adjudikasi', 'kelas_benar']:
        d[c] = d[c].fillna('').astype(str).str.strip()
    dua = d[(d['label_A'] != '') & (d['label_B'] != '')]
    d['label_final'] = label_final(d)
    sudah = d[d['label_final'].notna()].copy()
    sudah['benar'] = (sudah['label_final'] == 'BENAR').astype(int)
    hasil = {'n_baris': len(d), 'n_label_final': len(sudah), 'belum_final': int(d['label_final'].isna().sum()),
             'n_dua_anotator': len(dua), 'kappa': cohen_kappa(dua['label_A'], dua['label_B']) if len(dua) else float('nan')}
    tbl = []
    for s, x in sudah.groupby('strata'):
        lo, hi = wilson(int(x['benar'].sum()), len(x))
        tbl.append({'strata': s, 'N_populasi': int(x['N_strata'].iloc[0]), 'n_sampel': len(x),
                    'precision': round(x['benar'].mean(), 3), 'CI95_Wilson': f"{lo:.3f}-{hi:.3f}"})
    tbl = pd.DataFrame(tbl)
    if len(sudah) and sudah['strata'].nunique() == d['strata'].nunique():
        p, lo, hi = presisi_berbobot(sudah, rng)
        hasil.update({'precision': p, 'ci_lo': lo, 'ci_hi': hi})
    galat = sudah[sudah['label_final'] != 'BENAR'].groupby(['strata', 'label_final', 'kelas_benar']).size().reset_index(name='n')
    koreksi = None
    salah = sudah[sudah['label_final'] != 'BENAR']
    if nama == 'internal':
        salah = salah[salah['kelas_benar'] != '']
        if len(salah):
            cocok = salah.apply(lambda r: (r['kelas_benar'] == 'EKSTERNAL' and r['v2_kelas'].startswith('EKSTERNAL')) or
                                          (r['kelas_benar'] == 'INTERNAL' and r['v2_kelas'] == 'INTERNAL') or
                                          (r['kelas_benar'] == 'BUKAN_RUJUKAN' and r['v2_kelas'] == 'TIDAK_DITANGKAP'), axis=1)
            koreksi = (len(salah), int(cocok.sum()))
    elif len(salah):
        koreksi = (len(salah), int(salah['v2_kelas'].isin(['BEDA', 'V2_SAJA']).sum()))
    return hasil, tbl, galat, koreksi

def evaluasi_probe(d, rng, B=BOOTSTRAP_B):
    d = d.copy()
    kol = ['benar_total', 'tertangkap_v1', 'tertangkap_v2']
    for k in kol:
        for s in 'AB':
            d[f'{k}_{s}'] = pd.to_numeric(d[f'{k}_{s}'], errors='coerce')
    ada_B = d[[f'{k}_B' for k in kol]].notna().all(axis=1).any()
    for k in kol:
        d[k] = (d[f'{k}_A'] + d[f'{k}_B']) / 2 if ada_B else d[f'{k}_A']
    d = d.dropna(subset=kol)
    if d.empty:
        return None
    g = {s: x for s, x in d.groupby('strata')}
    N = {s: float(x['N_strata'].iloc[0]) for s, x in g.items()}
    def rasio(smp, k):
        bawah = sum(N[s] * x['benar_total'].mean() for s, x in smp.items())
        return sum(N[s] * x[k].mean() for s, x in smp.items()) / bawah if bawah else float('nan')
    hasil = {'n_chunk': len(d), 'dua_anotator': bool(ada_B)}
    if ada_B:
        hasil['kappa_benar_total'] = cohen_kappa(d['benar_total_A'], d['benar_total_B'])
    for k in ['tertangkap_v1', 'tertangkap_v2']:
        boot = [rasio({s: x.iloc[rng.integers(0, len(x), len(x))] for s, x in g.items()}, k) for _ in range(B)]
        hasil[k] = (rasio(g, k), float(np.nanpercentile(boot, 2.5)), float(np.nanpercentile(boot, 97.5)))
    return hasil

HASIL_AUDIT = {}
if not LEMBAR_AUDIT.exists():
    print("Lembar audit belum dibuat - jalankan sel pembuatan lembar terlebih dahulu.")
else:
    xl = pd.ExcelFile(LEMBAR_AUDIT)
    rng = np.random.default_rng(RANDOM_SEED)
    baris = []
    for nama in ['internal', 'eksternal']:
        d = xl.parse(nama)
        if d.empty or d[['label_A', 'label_adjudikasi']].isna().all().all():
            print(f"[{nama}] belum ada label - isi kolom label_A (dan label_B) pada lembar audit.")
            continue
        hasil, tbl, galat, koreksi = evaluasi_sheet(nama, d, rng)
        HASIL_AUDIT[nama] = hasil
        print(f"\n=== {nama.upper()} === (berlabel final {hasil['n_label_final']}/{hasil['n_baris']}; belum final {hasil['belum_final']})")
        display(tbl)
        if 'precision' in hasil:
            print(f"Precision v1 (berbobot strata): {hasil['precision']:.1%}  CI95 bootstrap [{hasil['ci_lo']:.1%} - {hasil['ci_hi']:.1%}]")
            baris.append([f'Rujukan {nama} (v1)', 'Precision berbobot strata', f"{hasil['precision']:.3f}",
                          f"{hasil['ci_lo']:.3f}-{hasil['ci_hi']:.3f}", hasil['n_label_final']])
        else:
            print("Estimasi berbobot belum dihitung: setiap strata harus punya minimal satu baris berlabel.")
        if hasil['n_dua_anotator']:
            print(f"Cohen's kappa A-B: {hasil['kappa']:.3f} ({tafsir_kappa(hasil['kappa'])}) pada {hasil['n_dua_anotator']} baris")
            baris.append([f'Rujukan {nama}', "Cohen's kappa (A vs B)", f"{hasil['kappa']:.3f}", tafsir_kappa(hasil['kappa']), hasil['n_dua_anotator']])
        else:
            print("Satu anotator: kappa tidak tersedia - tulis sebagai keterbatasan.")
        if len(galat):
            print("Diagnosis galat v1:"); display(galat)
        if koreksi:
            print(f"Dari {koreksi[0]} galat v1, {koreksi[1]} diubah ke arah yang sesuai oleh v2 (verifikasi manual bila 'eksternal').")
            baris.append([f'Rujukan {nama}', 'Galat v1 yang dikoreksi v2', f"{koreksi[1]}/{koreksi[0]}", '-', koreksi[0]])
    if 'probe' in xl.sheet_names:
        pr = evaluasi_probe(xl.parse('probe'), rng)
        if pr:
            HASIL_AUDIT['probe'] = pr
            print(f"\n=== PROBE RECALL === ({pr['n_chunk']} chunk)")
            for k, nm in [('tertangkap_v1', 'v1'), ('tertangkap_v2', 'v2')]:
                e, lo, hi = pr[k]
                print(f"Recall {nm} (estimasi, berbobot strata): {e:.1%}  CI95 [{lo:.1%} - {hi:.1%}]")
                baris.append([f'Rujukan (probe)', f'Recall {nm}', f'{e:.3f}', f'{lo:.3f}-{hi:.3f}', pr['n_chunk']])
        else:
            print("\n[probe] belum ada baris terisi.")
    if baris:
        ringkasan_audit = pd.DataFrame(baris, columns=['Komponen', 'Metrik', 'Nilai', 'CI95 / tafsir', 'n'])
        print("\n=== TABEL RINGKAS (siap disalin ke Bab 4) ===")
        display(ringkasan_audit)
        ringkasan_audit.to_csv(AUDIT_DIR / f'ringkasan_akurasi_{TIMESTAMP}.csv', index=False, encoding='utf-8-sig')
        with open(AUDIT_DIR / f'hasil_audit_{TIMESTAMP}.json', 'w', encoding='utf-8') as f:
            json.dump(HASIL_AUDIT, f, indent=2, ensure_ascii=False, default=float)
        print("Tersimpan di", AUDIT_DIR)


=== INTERNAL === (berlabel final 108/200; belum final 92)


,strata,N_populasi,n_sampel,precision,CI95_Wilson
0,int_tak_ditemukan,397,36,0.194,0.098-0.350
1,int_valid_sama,6553,24,0.625,0.427-0.788
2,int_valid_v1v2_beda,246,48,0.000,0.000-0.074


Precision v1 (berbobot strata): 58.0%  CI95 bootstrap [39.3% - 76.7%]
Cohen's kappa A-B: 0.363 (lemah) pada 200 baris
Diagnosis galat v1:


,strata,label_final,kelas_benar,n
0,int_tak_ditemukan,BUKAN_RUJUKAN,,14
1,int_tak_ditemukan,SALAH_JENIS,,15
2,int_valid_sama,BUKAN_RUJUKAN,,8
3,int_valid_sama,SALAH_JENIS,,1
4,int_valid_v1v2_beda,BUKAN_RUJUKAN,,8
5,int_valid_v1v2_beda,SALAH_JENIS,,40


[eksternal] belum ada label - isi kolom label_A (dan label_B) pada lembar audit.

[probe] belum ada baris terisi.

=== TABEL RINGKAS (siap disalin ke Bab 4) ===


,Komponen,Metrik,Nilai,CI95 / tafsir,n
0,Rujukan internal (v1),Precision berbobot strata,0.580,0.393-0.767,108
1,Rujukan internal,Cohen's kappa (A vs B),0.363,lemah,200


Tersimpan di /content/drive/MyDrive/Project 7/audit_rujukan


### Tahap 1d - Bangun ekstraksi norma deontik

Ekstraksi Norma Deontik Otomatis berbasis Rule-Based NLP & Regular Expressions yang presisi untuk hukum Indonesia. Kode ini mengekstraksi modalitas secara otomatis dari kolom isi serta mendeteksi frasa kunci penandanya.

In [10]:
# ==============================================================================
# TAHAP 1D: PIPELINE EKSTRAKSI NORMA DEONTIK PERBAIKAN & VALIDASI METRIK
# ==============================================================================
import re
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, cohen_kappa_score
from scipy.stats import chi2_contingency

# 1. PERBAIKAN POLA REGEX DENGAN PRINSIP KONTEKS DEONTIK HUKUM INDONESIA
# Perbaikan: Deteksi kombinasi sanksi, larangan, kewajiban, baru hak/izin.
POLA_DEONTIK_V2 = {
    # 1. Sanksi & Akibat Hukum (Mencegah "dapat dikenakan sanksi" masuk ke IZIN_HAK)
    'SANKSINORM': re.compile(
        r'\b(?:dikenakan|dikenai|dikenakan\s+sanksi|sanksi\s+administratif|pidana|denda|pencabutan)\b',
        re.I
    ),
    # 2. Larangan
    'LARANGAN': re.compile(
        r'\b(?:dilarang|tidak\s+boleh|tidak\s+diperkenankan|tidak\s+dapat\s+(?:melakukan|mengajukan|diberikan))\b',
        re.I
    ),
    # 3. Kewajiban (Abaikan kata 'patut' yang kontroversial)
    'KEWAJIBAN': re.compile(
        r'\b(?:wajib|harus|diharuskan|keharusan)\b',
        re.I
    ),
    # 4. Izin / Hak / Kewenangan (Pengecualian konteks sanksi)
    'IZIN_HAK': re.compile(
        r'\b(?:berhak|berwenang|diberikan\s+hak|diberi\s+izin)\b|'
        r'\bdapat\b(?!\s+(?:dikenakan|dikenai|diberikan\s+sanksi|dijatuhi))',
        re.I
    ),
    # 5. Deklaratif / Definisi
    'DEKLARATIF': re.compile(
        r'\b(?:merupakan|adalah|yang\s+dimaksud\s+dengan|meliputi|terdiri\s+atas)\b',
        re.I
    )
}

def ekstrak_norma_deontik_v2(teks: str) -> dict:
    """
    Ekstraksi norma deontik berbasis aturan kontekstual yang diperbaiki.
    """
    if not isinstance(teks, str) or not teks.strip():
        return {'deontik_otomatis': 'KETENTUAN_UMUM', 'deontik_marker': None}

    # Urutan Evaluasi Kontekstual yang Diperbaiki
    if POLA_DEONTIK_V2['SANKSINORM'].search(teks):
        m = POLA_DEONTIK_V2['SANKSINORM'].search(teks)
        return {'deontik_otomatis': 'SANKSI', 'deontik_marker': m.group().lower()}

    elif POLA_DEONTIK_V2['LARANGAN'].search(teks):
        m = POLA_DEONTIK_V2['LARANGAN'].search(teks)
        return {'deontik_otomatis': 'LARANGAN', 'deontik_marker': m.group().lower()}

    elif POLA_DEONTIK_V2['KEWAJIBAN'].search(teks):
        m = POLA_DEONTIK_V2['KEWAJIBAN'].search(teks)
        return {'deontik_otomatis': 'KEWAJIBAN', 'deontik_marker': m.group().lower()}

    elif POLA_DEONTIK_V2['IZIN_HAK'].search(teks):
        m = POLA_DEONTIK_V2['IZIN_HAK'].search(teks)
        return {'deontik_otomatis': 'IZIN_HAK', 'deontik_marker': m.group().lower()}

    elif POLA_DEONTIK_V2['DEKLARATIF'].search(teks):
        m = POLA_DEONTIK_V2['DEKLARATIF'].search(teks)
        return {'deontik_otomatis': 'DEKLARATIF', 'deontik_marker': m.group().lower()}

    else:
        return {'deontik_otomatis': 'KETENTUAN_UMUM', 'deontik_marker': None}

# 2. Terapkan Hasil Ekstraksi Otomatis ke Korpus
e_results = korpus['isi'].apply(ekstrak_norma_deontik_v2)
korpus['deontik_otomatis'] = [r['deontik_otomatis'] for r in e_results]
korpus['deontik_marker'] = [r['deontik_marker'] for r in e_results]

# 3. TAMPILKAN TABEL SILANG KONTINGENSI HASSIL REVISI
ct = pd.crosstab(korpus['tipe'], korpus['deontik_otomatis'], margins=True)
print("=== TABEL SILANG KONTINGENSI (REVISI V2) ===")
display(ct)

# 4. MAPPING PEMETAN KELAS UNTUK EVALUASI KESEPAKATAN STATISTIK
# Menyelaraskan kelas 'tipe' manual dengan kelas 'deontik_otomatis'
MAPPING_TIPE = {
    'Definisi': 'DEKLARATIF',
    'Hak': 'IZIN_HAK',
    'Ketentuan': 'KETENTUAN_UMUM',
    'Kewajiban': 'KEWAJIBAN',
    'Larangan': 'LARANGAN',
    'Sanksi': 'SANKSI'
}

y_true = korpus['tipe'].map(MAPPING_TIPE).fillna('KETENTUAN_UMUM')
y_pred = korpus['deontik_otomatis']

# 5. KALKULASI METRIK KESEPAKATAN STATISTIK
def hitung_cramers_v(contingency_table):
    chi2 = chi2_contingency(contingency_table)[0]
    n = contingency_table.sum().sum()
    r, k = contingency_table.shape
    return np.sqrt(chi2 / (n * (min(k, r) - 1)))

ct_clean = pd.crosstab(korpus['tipe'], korpus['deontik_otomatis'])
kappa = cohen_kappa_score(y_true, y_pred)
cramer_v = hitung_cramers_v(ct_clean)

print("\n=== METRIK KESEPAKATAN & VALIDASI ===")
print(f"1. Cohen's Kappa Score      : {kappa:.4f}")
print(f"2. Cramér's V Association    : {cramer_v:.4f}")

print("\n=== LAPORAN EVALUASI PER KELAS (CLASSIFICATION REPORT) ===")
print(classification_report(y_true, y_pred, zero_division=0))

=== TABEL SILANG KONTINGENSI (REVISI V2) ===


deontik_otomatis,DEKLARATIF,IZIN_HAK,KETENTUAN_UMUM,KEWAJIBAN,LARANGAN,SANKSI,All
tipe,,,,,,,
Korea (Rep.),0,0,1,0,0,0,1
4 SATELIT BERGERAK (Bumi ke SATELIT BERGERAK (Bumi ke ruang angkasa) 5.351A ruang angkasa) 5.351A 5.379B 5.379C 5.379B 5.379C ASTRONOMI RADIO ASTRONOMI RADIO PENELITIAN RUANG ANGKASA PENELITIAN RUANG ANGKASA (pasif) (pasif) Tetap Tetap Bergerak kecuali bergerak Bergerak kecuali bergerak penerbangan penerbangan Bantuan meteorologis 5.379 5.149 5.341 5.379 5.379A 5.149 5.341 5.379A INS18 1668,0,0,1,0,0,0,1
400,0,0,1,0,0,0,1
Definisi,1020,159,11,82,14,49,1335
Hak,22,257,189,0,0,11,479
Ketentuan,1612,1256,17327,2,4,196,20397
Kewajiban,23,44,273,1550,2,37,1929
Larangan,0,0,9,0,257,3,269
Sanksi,4,33,220,21,0,1801,2079



=== METRIK KESEPAKATAN & VALIDASI ===
1. Cohen's Kappa Score      : 0.6481
2. Cramér's V Association    : 0.7330

=== LAPORAN EVALUASI PER KELAS (CLASSIFICATION REPORT) ===
                precision    recall  f1-score   support

    DEKLARATIF       0.38      0.76      0.51      1335
      IZIN_HAK       0.15      0.54      0.23       479
KETENTUAN_UMUM       0.96      0.85      0.90     20400
     KEWAJIBAN       0.94      0.80      0.86      1929
      LARANGAN       0.93      0.96      0.94       269
        SANKSI       0.86      0.87      0.86      2079

      accuracy                           0.84     26491
     macro avg       0.70      0.80      0.72     26491
  weighted avg       0.91      0.84      0.86     26491



# Knowledge Graph

In [11]:
# ---------------- Pilih ekstraksi yang dipakai membangun KG ----------------
if EKSTRAKSI_AKTIF == 'v2':
    ref_kg_internal = ref_internal_v2[ref_internal_v2['valid'] == True].copy()   # noqa: E712
    ref_kg_eksternal = ref_eksternal_v2[ref_eksternal_v2['dalam_korpus'] & ~ref_eksternal_v2['rujukan_diri']].copy()
else:
    ref_kg_internal = ref_internal_v1[ref_internal_v1['valid']].assign(doc_tuju=lambda t: t['doc_id']).copy()
    ref_kg_eksternal = ref_eksternal_v1[ref_eksternal_v1['dalam_korpus']].copy()
print(f"KG akan memakai ekstraksi {EKSTRAKSI_AKTIF}: {len(ref_kg_internal)} rujukan antarpasal, "
      f"{len(ref_kg_eksternal)} rujukan antardokumen (sebelum deduplikasi).")

KG akan memakai ekstraksi v1: 6799 rujukan antarpasal, 106 rujukan antardokumen (sebelum deduplikasi).


In [12]:
KG = nx.MultiDiGraph()

def id_bab(doc, bab):     return f"{doc}::{bab}"
def id_pasal(doc, pasal): return f"{doc}::P{pasal}"

# 1) Hirarki struktural (idempoten: key tetap mencegah sisi paralel) &
# Perbarui bagian penambahan node satuan pada pembuatan KG
for _, r in korpus.iterrows():
    d = r['doc_id']
    b = id_bab(d, r['bab'])
    p = id_pasal(d, r['pasal'])
    tahun_value = int(r['tahun']) if pd.notna(r['tahun']) else None

    KG.add_node(d, tipe='dokumen', label=r['nama_dokumen'], jenis=r['jenis'], tahun=tahun_value)
    KG.add_node(b, tipe='bab', label=str(r['nama_bab']))
    KG.add_node(p, tipe='pasal', label='Pasal ' + str(r['pasal']))

    # Menambahkan properti deontik otomatis ke simpul satuan
    KG.add_node(
        r['chunk_id'],
        tipe='satuan',
        norma=r['tipe'],
        norma_deontik_auto=r['deontik_otomatis'], # Hasil Ekstraksi Otomatis
        deontik_marker=r['deontik_marker'],       # Kata kunci pencetus
        pasal=str(r['pasal'])
    )

    KG.add_edge(d, b, key='memuat', rel='memuat')
    KG.add_edge(b, p, key='memuat', rel='memuat')
    KG.add_edge(p, r['chunk_id'], key='memuat', rel='memuat')

# 2) Rujukan antarpasal (hasil ekstraksi aktif; satu sisi per pasangan pasal; key mencegah sisi ganda)
print("Ekstraksi aktif untuk KG:", EKSTRAKSI_AKTIF)
pasangan = (ref_kg_internal.drop_duplicates(['doc_id', 'pasal_asal', 'doc_tuju', 'pasal_tuju']))
for _, r in pasangan.iterrows():
    KG.add_edge(id_pasal(r['doc_id'], r['pasal_asal']),
                id_pasal(r['doc_tuju'], r['pasal_tuju']),
                key='merujuk', rel='merujuk')

# 3) Rujukan antar-dokumen (dokumen -> dokumen, hanya yang tujuannya ada dalam korpus)
sub = ref_kg_eksternal.drop_duplicates(['doc_id', 'doc_tuju'])
for _, r in sub.iterrows():
    if r['doc_id'] != r['doc_tuju']:
        KG.add_edge(r['doc_id'], r['doc_tuju'], key='merujuk_dokumen', rel='merujuk_dokumen')

# 4) Entitas (key mencegah sisi ganda bila entitas sama muncul dua kali)
for _, r in korpus[korpus['entitas'].notna()].iterrows():
    for ent in str(r['entitas']).split(','):
        ent = ent.strip()
        if ent:
            eid = 'ENT::' + ent
            KG.add_node(eid, tipe='entitas', label=ent)
            KG.add_edge(r['chunk_id'], eid, key='menyebut', rel='menyebut')

print(f"Simpul: {KG.number_of_nodes():,} | Sisi: {KG.number_of_edges():,}")
print(pd.Series([d['tipe'] for _, d in KG.nodes(data=True)]).value_counts().to_string())
print()
print(pd.Series([d['rel'] for _, _, d in KG.edges(data=True)]).value_counts().to_string())

with open(os.path.join(IFACT_DIR, 'knowledge_graph.gpickle'), 'wb') as f:
    pickle.dump(KG, f)

Ekstraksi aktif untuk KG: v1
Simpul: 29,347 | Sisi: 47,070
satuan     26491
pasal       2577
bab          217
entitas       36
dokumen       26

memuat             31577
menyebut           10546
merujuk             4925
merujuk_dokumen       22


In [13]:
try:
    from pyvis.network import Network
except ImportError:
    %pip install -q pyvis
    from pyvis.network import Network

# Statistik graf untuk laporan tesis + visualisasi sampel
deg = pd.Series({n: KG.degree(n) for n, d in KG.nodes(data=True) if d['tipe'] == 'pasal'})
print("Pasal paling terhubung (hub regulasi):")
for n, v in deg.sort_values(ascending=False).head(8).items():
    print(f"  {n}  (derajat {v})")

# Visualisasi interaktif subgraf sekitar satu pasal (buka file HTML hasilnya)
def gambar_subgraf(node_pusat: str, radius: int = 1, out='subgraf.html'):
    simpul = {node_pusat}
    for _ in range(radius):
        tetangga = set()
        for s in simpul:
            tetangga |= set(KG.successors(s)) | set(KG.predecessors(s))
        simpul |= tetangga
    sub = KG.subgraph(simpul)
    warna = {'dokumen':'#d62728','bab':'#ff7f0e','pasal':'#1f77b4','satuan':'#2ca02c','entitas':'#9467bd'}
    net = Network(height='600px', directed=True, notebook=False, cdn_resources='remote')
    for n, d in sub.nodes(data=True):
        net.add_node(n, label=d.get('label', n)[:40], color=warna[d['tipe']], title=d['tipe'])
    for u, v, d in sub.edges(data=True):
        net.add_edge(u, v, label=d['rel'])
    net.save_graph(out); print("Tersimpan:", out)

gambar_subgraf(deg.idxmax(), radius=1)

Pasal paling terhubung (hub regulasi):
  Perdirjen_07_2021::P75  (derajat 712)
  UU_6_2023::P1  (derajat 593)
  Permen_8_2026::P9  (derajat 336)
  UU_6_2023::P15  (derajat 207)
  UU_6_2023::P26  (derajat 202)
  UU_6_2023::P7  (derajat 179)
  UU_6_2023::P5  (derajat 177)
  Permen_2_2025::P1  (derajat 164)
Tersimpan: subgraf.html


In [14]:
G_rujuk = nx.DiGraph([(u, v) for u, v, d in KG.edges(data=True) if d['rel'] == 'merujuk'])
masuk  = pd.Series(dict(G_rujuk.in_degree())).sort_values(ascending=False)
keluar = pd.Series(dict(G_rujuk.out_degree())).sort_values(ascending=False)

print('Pasal paling banyak dirujuk:')
for n, v in masuk.head(8).items():
    print('  ', n, '(dirujuk', v, 'kali)')
print('Pasal paling banyak merujuk:')
for n, v in keluar.head(8).items():
    print('  ', n, '(merujuk', v, 'kali)')

gambar_subgraf(masuk.index[0], radius=1)

Pasal paling banyak dirujuk:
   Perdirjen_3_2024::P5 (dirujuk 48 kali)
   UU_6_2023::P1 (dirujuk 36 kali)
   Permen_9_2023::P5 (dirujuk 30 kali)
   UU_6_2023::P9 (dirujuk 27 kali)
   Perppu_2_2022::P26 (dirujuk 27 kali)
   Perppu_2_2022::P17 (dirujuk 27 kali)
   UU_6_2023::P17 (dirujuk 26 kali)
   Perppu_2_2022::P1O (dirujuk 26 kali)
Pasal paling banyak merujuk:
   UU_6_2023::P85 (merujuk 35 kali)
   Perppu_2_2022::P85 (merujuk 34 kali)
   UU_6_2023::P15 (merujuk 29 kali)
   Perppu_2_2022::P47 (merujuk 28 kali)
   Perppu_2_2022::P48 (merujuk 28 kali)
   Perppu_2_2022::P15 (merujuk 27 kali)
   UU_6_2023::P47 (merujuk 27 kali)
   Perdirjen_07_2021::P75 (merujuk 27 kali)
Tersimpan: subgraf.html


### Visualisasi Hubungan Antar Dokumen (Gabungan)

Kode di bawah ini memfilter simpul bertipe `dokumen` dan `pasal` yang memiliki relasi lintas dokumen, sehingga Anda bisa melihat keterkaitan antara **Perdirjen_3_2024** dan **PP_28_2025** secara langsung.

In [15]:
# Membuat subgraf yang berisi simpul dokumen dan pasal-pasal utama dari kedua dokumen
simpul_terpilih = set()

# Masukkan semua simpul tipe 'dokumen'
for n, d in KG.nodes(data=True):
    if d.get('tipe') == 'dokumen':
        simpul_terpilih.add(n)

# Masukkan pasal-pasal yang memiliki derajat keterhubungan tinggi (top 15)
deg_pasal = pd.Series({n: KG.degree(n) for n, d in KG.nodes(data=True) if d['tipe'] == 'pasal'})
top_pasal = deg_pasal.sort_values(ascending=False).head(15).index
simpul_terpilih.update(top_pasal)

# Tambahkan tetangga dari pasal-pasal top tersebut
for p in top_pasal:
    simpul_terpilih.update(KG.successors(p))
    simpul_terpilih.update(KG.predecessors(p))

# Filter graf hanya untuk simpul terpilih
sub_gabungan = KG.subgraph(simpul_terpilih)

# Visualisasikan
warna = {'dokumen':'#d62728', 'bab':'#ff7f0e', 'pasal':'#1f77b4', 'satuan':'#2ca02c', 'entitas':'#9467bd'}
net_gabungan = Network(height='700px', width='100%', directed=True, notebook=False, cdn_resources='remote')

for n, d in sub_gabungan.nodes(data=True):
    net_gabungan.add_node(n, label=d.get('label', n)[:40], color=warna.get(d.get('tipe'), '#97c2fc'), title=d.get('tipe'))

for u, v, d in sub_gabungan.edges(data=True):
    net_gabungan.add_edge(u, v, label=d['rel'])

net_gabungan.save_graph('subgraf_gabungan.html')
print("Tersimpan: subgraf_gabungan.html")

Tersimpan: subgraf_gabungan.html


In [16]:
import networkx as nx
import pandas as pd

# Menggunakan objek KG (Knowledge Graph) yang telah didefinisikan sebelumnya

print("=== RINGKASAN KNOWLEDGE GRAPH ===")
print("Tipe graf       :", type(KG).__name__)
print("Jumlah node     :", KG.number_of_nodes())
print("Jumlah edge     :", KG.number_of_edges())
print("Jumlah komponen terhubung:",
      nx.number_weakly_connected_components(KG)
      if KG.is_directed() else nx.number_connected_components(KG))

# Distribusi tipe node
node_types = pd.Series([
    attrs.get("tipe", "unknown")  # Diubah dari 'type' ke 'tipe' sesuai definisi di cell e02052d6
    for _, attrs in KG.nodes(data=True)
]).value_counts()

print("\nDistribusi tipe node:")
print(node_types)

# Distribusi tipe relasi
edge_types = pd.Series([
    attrs.get("rel", "unknown")   # Diubah dari 'relation'/'label' ke 'rel' sesuai definisi di cell e02052d6
    for _, _, attrs in KG.edges(data=True)
]).value_counts()

print("\nDistribusi tipe relasi:")
print(edge_types)

# Node tanpa edge
isolated = list(nx.isolates(KG))
print("\nJumlah isolated nodes:", len(isolated))

# Duplikasi edge berdasarkan pasangan sumber-tujuan-jenis relasi
edge_records = []
for u, v, key, attrs in KG.edges(keys=True, data=True):
    relation = attrs.get("rel", "unknown")
    edge_records.append((u, v, relation))

edge_df = pd.DataFrame(
    edge_records, columns=["source", "target", "relation"]
)

duplicates = edge_df[
    edge_df.duplicated(
        subset=["source", "target", "relation"],
        keep=False
    )
].sort_values(["source", "target", "relation"])

print("Jumlah edge dalam kelompok duplikat:",
      len(duplicates))
display(duplicates.head(20))

=== RINGKASAN KNOWLEDGE GRAPH ===
Tipe graf       : MultiDiGraph
Jumlah node     : 29347
Jumlah edge     : 47070
Jumlah komponen terhubung: 4

Distribusi tipe node:
satuan     26491
pasal       2577
bab          217
entitas       36
dokumen       26
Name: count, dtype: int64

Distribusi tipe relasi:
memuat             31577
menyebut           10546
merujuk             4925
merujuk_dokumen       22
Name: count, dtype: int64

Jumlah isolated nodes: 0
Jumlah edge dalam kelompok duplikat: 0


,source,target,relation


## Tahap 3 Demonstrasi Dampak (Graph-RAG PoC)
Implementasikan Proof-of-Concept (PoC) Graph-RAG (Retrieval-Augmented Generation berbasis Graf) sederhana.

In [17]:
# ==============================================================================
# TAHAP 3: DEMONSTRASI DAMPAK (GRAPH-RAG VS NAIVE RETRIEVAL)
# ==============================================================================
import networkx as nx
import pandas as pd

def naive_retrieval(korpus_df, query_keywords, top_k=3):
    """
    Simulasi Naive Retrieval berbasis pencarian kata kunci / BM25 sederhana.
    """
    pattern = '|'.join(query_keywords)
    matched = korpus_df[korpus_df['isi'].str.contains(pattern, case=False, na=False)]
    return matched.head(top_k)

def graph_rag_retrieval(kg_graph, korpus_df, query_keywords, top_k=3):
    """
    Graph-RAG Retrieval: Mengambil chunk relevan lalu melacak relasi
    rujukan (merujuk/sanksi) pada Knowledge Graph.
    """
    # 1. Langkah awal: temukan chunk acuan (anchor)
    baseline_hits = naive_retrieval(korpus_df, query_keywords, top_k=top_k)
    anchor_chunk_ids = set(baseline_hits['chunk_id'].tolist())

    expanded_chunk_ids = set(anchor_chunk_ids)
    context_edges = []

    # 2. Graph Traversal: Melacak tetangga pada KG (Pasal Rujukan & Sanksi terkait)
    for chunk_id in anchor_chunk_ids:
        if chunk_id not in kg_graph:
            continue

        # Dapatkan node pasal induk dari chunk
        pasal_nodes = [u for u, v, d in kg_graph.in_edges(chunk_id, data=True) if d.get('rel') == 'memuat']

        for p_node in pasal_nodes:
            # Traversal edge 'merujuk' (rujukan antar pasal)
            for u, v, d in kg_graph.out_edges(p_node, data=True):
                if d.get('rel') == 'merujuk':
                    # Ambil chunk milik pasal tujuan
                    target_chunks = [v_c for u_c, v_c, d_c in kg_graph.out_edges(v, data=True) if d_c.get('rel') == 'memuat']
                    expanded_chunk_ids.update(target_chunks)
                    context_edges.append((p_node, 'merujuk', v))

            # Traversal edge 'merujuk' masuk (misal pasal sanksi yang merujuk ke pasal ini)
            for u, v, d in kg_graph.in_edges(p_node, data=True):
                if d.get('rel') == 'merujuk':
                    target_chunks = [v_c for u_c, v_c, d_c in kg_graph.out_edges(u, data=True) if d_c.get('rel') == 'memuat']
                    expanded_chunk_ids.update(target_chunks)
                    context_edges.append((u, 'merujuk', p_node))

    # 3. Ambil seluruh konteks pasal/chunk terluas dari KG
    res_df = korpus_df[korpus_df['chunk_id'].isin(expanded_chunk_ids)].copy()
    return res_df, context_edges

# --- EKSEKUSI DEMONSTRASI ---
QUERY_TEST = ["izin usaha", "persyaratan dasar"]

print("=== 1. NAIVE RETRIEVAL (TANPA KG) ===")
naive_res = naive_retrieval(korpus, QUERY_TEST, top_k=2)
for _, r in naive_res.iterrows():
    print(f"[{r['chunk_id']}] ({r['tipe']}) -> {r['isi'][:120]}...\n")

print("="*60)
print("=== 2. GRAPH-RAG RETRIEVAL (MEMANFAATKAN STRUKTUR KG & RUJUKAN) ===")
grag_res, edges_traversed = graph_rag_retrieval(KG, korpus, QUERY_TEST, top_k=2)

print(f"Total Chunk Berhasil Diperluas via Graf: {len(grag_res)} (dari {len(naive_res)} chunk awal)")
print(f"Relasi Graf Yang Dilintasi: {edges_traversed}\n")

print("Sampel Tambahan Konteks dari Graph-RAG (Pasal Sanksi/Rujukan Terkait):")
for _, r in grag_res.iterrows():
    deontik = r.get('deontik_otomatis', 'N/A')
    print(f"[{r['chunk_id']}] [Deontik: {deontik}] -> {r['isi'][:120]}...\n")

=== 1. NAIVE RETRIEVAL (TANPA KG) ===
[PP_28_2025__P2__a2] (Ketentuan) -> Ruang lingkup penyelenggaraan PBBR sebagaimana dimaksud pada ayat (1) meliputi: a. persyaratan dasar; b. PB; C. PB UMKU;...

[PP_28_2025__P3__ha] (Ketentuan) -> pelaksanaan penerbitan persyaratan dasar, PB, dan PB UMKU secara lebih efektif dan sederhana; dan...

=== 2. GRAPH-RAG RETRIEVAL (MEMANFAATKAN STRUKTUR KG & RUJUKAN) ===
Total Chunk Berhasil Diperluas via Graf: 51 (dari 2 chunk awal)
Relasi Graf Yang Dilintasi: [('PP_28_2025::P3', 'merujuk', 'PP_28_2025::P2'), ('PP_28_2025::P5', 'merujuk', 'PP_28_2025::P2'), ('PP_28_2025::P20', 'merujuk', 'PP_28_2025::P2'), ('PP_28_2025::P211', 'merujuk', 'PP_28_2025::P2'), ('PP_28_2025::P215', 'merujuk', 'PP_28_2025::P2'), ('PP_28_2025::P3', 'merujuk', 'PP_28_2025::P2'), ('PP_28_2025::P31', 'merujuk', 'PP_28_2025::P3')]

Sampel Tambahan Konteks dari Graph-RAG (Pasal Sanksi/Rujukan Terkait):
[PP_28_2025__P2__a1] [Deontik: KETENTUAN_UMUM] -> Pemerintah Pusat menetapkan keb

## Kesimpulan & Keterbatasan

### 1. Kesimpulan
Berdasarkan hasil konstruksi dan evaluasi *Knowledge Graph* (KG) regulasi telekomunikasi Indonesia pada RQ1, beberapa poin penting dapat disimpulkan:
* **Otomatisasi & Skalabilitas:** Metodologi berbasis aturan (*Rule-Based NLP*) berhasil mengonstruksi graf pengetahuan berskala besar secara otomatis dari dokumen mentah PDF/CSV dengan total **29.347 simpul (nodes)** dan **47.070 sisi (edges)**.
* **Akurasi Ekstraksi (v1):** Hasil audit manual dengan metode sampling terstrata menunjukkan bahwa rujukan internal versi awal (v1) memiliki **presisi berbobot sebesar 58,0% (CI95%: 39,3% - 76,7%)**. Hal ini mengonfirmasi adanya tantangan struktural yang signifikan dalam dokumen hukum Indonesia yang tidak bisa diselesaikan secara naif.
* **Kesepakatan Anotator:** Pengukuran *Cohen's Kappa* antar-anotator independen menghasilkan skor **0,363 (kategori lemah)**, menunjukkan adanya ambiguitas penafsiran teks rujukan (misalnya rujukan implisit atau rujukan ke lampiran/dokumen eksternal tanpa nomor).
* **Peningkatan ke v2:** Dari hasil diagnosis galat v1, mayoritas kesalahan disebabkan oleh rujukan ke regulasi lain (*cross-regulation*) yang salah diidentifikasi sebagai rujukan internal karena format penulisan "Pasal N". Ekstraksi v2 berhasil mengoreksi kesalahan ini secara signifikan.

### 2. Keterbatasan Penelitian
* **Ketergantungan Struktur Korpus:** Validasi struktural otomatis sangat bergantung pada kelengkapan korpus aktif. Rujukan ke dokumen di luar korpus (seperti UU No. 8 Tahun 1981 tentang KUHAP yang sering dirujuk) dicatat sebagai *tidak ditemukan* atau *tidak dapat dicek*.
* **Ambiguitas Sintaksis Bahasa Hukum:** Pola linguistik regulasi Indonesia sering kali menggunakan struktur melingkar, perujukan bersyarat, serta rentang pasal yang kompleks. Hal ini membutuhkan aturan kontekstual yang sangat spesifik dan belum sepenuhnya terakomodasi sempurna tanpa bantuan model bahasa besar (LLM).
* **Keterbatasan Evaluasi Probe Recall:** Evaluasi *recall* masih terbatas pada sampel kecil (*probe chunks*) karena tingginya biaya anotasi manual pada keseluruhan dokumen.